# Selection Overlaps

How much the selections plotted in `open_data_distributions.ipynb` overlap with each other.  The quantity everywhere
is the **weighted total prediction scaled to the full runs 1-5 data POT**: the summed `wc_net_weight_full_pred` of
every prediction event (MC + EXT, BDT test events scaled up by the test fraction exactly as in
`open_data_distributions.ipynb`; the full-dataset POT per run period is `expected_full_dataset_data_POT` in
`src/pot_and_trigger_numbers.py`).

Selections are always kept separate in $Np$/$0p$ and $0\mu$/$1\mu$.  The WC NC $\Delta$, inclusive $1\gamma$, and
inclusive $\nu_e$CC selections are split into $Np$/$0p$ with the same reco proton cut
(`wc_reco_num_protons_35_MeV` $> 0$ or $= 0$); the WC selections have no muon split, so they are compared against the
$0\mu$ and $1\mu$ BAGELS selections separately.

The BAGELS selections (the multiclass-BDT reco categories, `src/reco_categories.py`) are mutually exclusive by
construction, so the comparisons here are the previously existing WC cut-based selections (NC
$\Delta\rightarrow N\gamma$, inclusive $1\gamma$, inclusive $\nu_e$CC) vs the BAGELS selections.

Plots (from `src/area_proportional_venn.py`), saved as PNG and PDF under `plots/selection_overlaps/`:
* **circle diagrams** (3-4 selections): the circles are fitted so every region's area matches its weight
  (eulerr-style).  Circles can't represent every 3+ set configuration exactly, so the fit error (`diagError`, the
  largest region-area-fraction mismatch) is printed below each diagram and the fitted areas are listed next to the
  true weights in each table.  Overlaps smaller than `min_shown_overlap` events are left out, with a note under the
  diagram; if a larger overlap can't be drawn with circles, that diagram's cutoff is raised just above it so the
  note stays exact.  The diagrams have no titles (they are described in the note text);
* **overlap matrices**: the fraction of each row selection's weighted events that also pass each column selection.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.file_locations import intermediate_files_location
from src.df_helpers import select_bdt_test_events
from src.signal_categories import train_category_labels
from src.reco_categories import reco_category_expr
from src.area_proportional_venn import (region_weights_from_membership, all_patterns, fmt_weight,
                                        draw_venn_fitted, draw_overlap_matrix)

training = "all_vars_2026_11_04"
reco_categories = train_category_labels
prob_categories = ["prob_" + cat for cat in reco_categories]
weight_var = "wc_net_weight_full_pred"

# overlap regions with fewer weighted events than this are left out of the circle diagrams (raised per diagram
# if a larger overlap can't be drawn with circles); a note under each diagram gives the cutoff actually used
min_shown_overlap = 5

plots_dir = "../plots/selection_overlaps"
os.makedirs(plots_dir, exist_ok=True)

def save(fig, name):
    fig.savefig(f"{plots_dir}/{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(f"{plots_dir}/{name}.pdf", bbox_inches="tight")

# File Loading

Same prediction sample as `open_data_distributions.ipynb`: generic neutrino preselection (`wc_kine_reco_Enu > 0`),
the same prediction samples, and only BDT test (scaled up) or never-trained events.  Only the columns needed for
the selections are loaded.

In [ ]:
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
pred_df = pl.scan_parquet(f"../training_outputs/{training}/predictions.parquet")
merged = all_df.join(pred_df, on=["filetype", "run", "subrun", "event"], how="left")

needed_vars = [
    "filetype", "run", "subrun", "event", "bdt_role", "bdt_prediction_weight_factor", weight_var,
    "wc_kine_reco_Enu", "wc_nc_delta_score", "wc_nue_score", "wc_reco_num_protons_35_MeV",
    "wc_shw_sp_n_20mev_showers", "wc_shw_sp_n_20br1_showers", "wc_reco_nuvtxX",
    "wc_single_photon_numu_score", "wc_single_photon_other_score",
    "wc_single_photon_ncpi0_score", "wc_single_photon_nue_score",
] + prob_categories
merged = merged.select(needed_vars).filter(pl.col("wc_kine_reco_Enu") > 0)

# same prediction definition as open_data_distributions.ipynb
pred = merged.filter(~pl.col("filetype").is_in(["data", "isotropic_one_gamma_overlay", "delete_one_gamma_overlay",
                                                "nuwro_fake_data", "fullosc_overlay"]))
pred = select_bdt_test_events(pred, [weight_var]).filter(pl.col(weight_var).is_not_null())

df = pred.with_columns([pl.col(p).fill_null(-1) for p in prob_categories]).collect()
del all_df, pred_df, merged, pred
weights = df[weight_var].to_numpy()
print(f"{df.height} prediction events, weighted total {weights.sum():.1f} (full data POT)")

# Selection Definitions

* **WC**: the previously existing cut-based selections, with the same cuts as `open_data_distributions.ipynb`.
* **BAGELS**: the nominal (mutually exclusive) multiclass-BDT reco categories.

In [ ]:
erin_inclusive_1g_expr = (
    (pl.col("wc_shw_sp_n_20mev_showers") > 0)
    & (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0)
    & (pl.col("wc_single_photon_numu_score") > 0.4)
    & (pl.col("wc_single_photon_other_score") > 0.2)
    & (pl.col("wc_single_photon_ncpi0_score") > -0.05)
    & (pl.col("wc_single_photon_nue_score") > -1.0)
    & (pl.col("wc_shw_sp_n_20br1_showers") == 1)
)
nc_delta = pl.col("wc_nc_delta_score") > 2.61
wc_nue = pl.col("wc_nue_score") > 7
Np, zero_p = pl.col("wc_reco_num_protons_35_MeV") > 0, pl.col("wc_reco_num_protons_35_MeV") == 0

reco_category = reco_category_expr(reco_categories)

# name: (polars expression, plot label)
selections = {
    "wc_nc_delta_Np":     (nc_delta & Np, r"WC NC$\Delta$ $Np$"),
    "wc_nc_delta_0p":     (nc_delta & zero_p, r"WC NC$\Delta$ $0p$"),
    "wc_inclusive_1g_Np": (erin_inclusive_1g_expr & Np, r"WC incl. $1\gamma$ $Np$"),
    "wc_inclusive_1g_0p": (erin_inclusive_1g_expr & zero_p, r"WC incl. $1\gamma$ $0p$"),
    "wc_nueCC_Np":        (wc_nue & Np, r"WC $\nu_e$CC $Np$"),
    "wc_nueCC_0p":        (wc_nue & zero_p, r"WC $\nu_e$CC $0p$"),
    **{f"bdt_{cat}": (reco_category == cat, f"BAGELS {cat}") for cat in reco_categories},
    "bdt_unselected":     (reco_category == "unselected", "BAGELS unselected"),
}
labels = {name: lab for name, (_, lab) in selections.items()}

member_df = df.select([expr.fill_null(False).alias(name) for name, (expr, _) in selections.items()])
membership = {name: member_df[name].to_numpy() for name in selections}

print(f"{'selection':<26} {'prediction (full data POT)':>27}")
for name in selections:
    print(f"{name:<26} {weights[membership[name]].sum():>27.1f}")

# Circle Diagram Helper

`compare(names)` computes the exclusive overlap regions (prediction weight), draws the fitted circle diagram,
and prints the fit error and the region table (true and fitted weight of every region).

In [ ]:
def compare(names, savename):
    mem = np.column_stack([membership[n] for n in names])
    pred_regions = region_weights_from_membership(mem, weights)

    fig, ax = plt.subplots(figsize=(9, 7))
    fit = draw_venn_fitted(ax, pred_regions, [labels[n] for n in names], min_shown=min_shown_overlap)
    save(fig, savename)
    plt.show()

    print(f"circle fit diagError = {100 * fit['diag_error']:.2f}% of the union area")
    if fit["hidden"]:
        print(f"overlaps with less than {fit['min_shown']:g} events are not shown (marked 'hidden' below)")
    print(f"{'region':<80} {'prediction':>11} {'circle fit':>11}")
    for k, p in enumerate(all_patterns(len(names))):
        desc = " & ".join(n for n, b in zip(names, p) if b)
        if not all(p):
            desc += "  (not " + ", ".join(n for n, b in zip(names, p) if not b) + ")"
        print(f"{desc:<80} {pred_regions[p]:>11.2f} {fit['fitted'][k]:>11.2f}" + ("  hidden" if p in fit["hidden"] else ""))

# WC vs BAGELS

Fitted circles.  The WC $Np$/$0p$ pairs against the BAGELS $Np$/$0p$ pairs (for WC inclusive $1\gamma$, against both the $0\mu$ and the
$1\mu$ BAGELS $1\gamma$ selections) (shows the $Np \leftrightarrow 0p$ migration between the two).

In [ ]:
multi_set_comparisons = [
    ["wc_nc_delta_Np", "wc_nc_delta_0p", "bdt_1gNp", "bdt_1g0p"],
    ["wc_inclusive_1g_Np", "wc_inclusive_1g_0p", "bdt_1gNp", "bdt_1g0p"],
    ["wc_inclusive_1g_Np", "wc_inclusive_1g_0p", "bdt_1gNp1mu", "bdt_1g0p1mu"],
    ["wc_nueCC_Np", "wc_nueCC_0p", "bdt_nueCC_Np", "bdt_nueCC_0p"],
]
for names in multi_set_comparisons:
    compare(names, "_".join(names).replace("bdt_", "bagels_"))

# Overlap Matrices

Row-normalized: each cell is the fraction of the row selection's weighted prediction that also passes the column
selection (with the overlap weight below it).

1. The WC selections and the corresponding BAGELS selections against each other.
2. Where each WC selection's events end up in the BAGELS categorization (each row sums to 100%, since every
   preselected event gets exactly one BAGELS category or "unselected").

In [ ]:
main_names = ["wc_nc_delta_Np", "wc_nc_delta_0p", "wc_inclusive_1g_Np", "wc_inclusive_1g_0p", "wc_nueCC_Np", "wc_nueCC_0p",
              "bdt_1gNp", "bdt_1g0p", "bdt_1gNp1mu", "bdt_1g0p1mu", "bdt_nueCC_Np", "bdt_nueCC_0p"]
fig, ax = plt.subplots(figsize=(12, 10.5))
im = draw_overlap_matrix(ax, np.column_stack([membership[n] for n in main_names]), weights,
                         [labels[n] for n in main_names],
                         title="Selection overlap fractions (weighted prediction, full data POT)")
fig.colorbar(im, ax=ax, fraction=0.04, label="fraction of row selection")
save(fig, "overlap_matrix_wc_and_bagels")
plt.show()

In [ ]:
wc_names = ["wc_nc_delta_Np", "wc_nc_delta_0p", "wc_inclusive_1g_Np", "wc_inclusive_1g_0p", "wc_nueCC_Np", "wc_nueCC_0p"]
bdt_names = [f"bdt_{cat}" for cat in reco_categories] + ["bdt_unselected"]

inter = np.array([[weights[membership[a] & membership[b]].sum() for b in bdt_names] for a in wc_names])
frac = inter / np.array([weights[membership[a]].sum() for a in wc_names])[:, None]

fig, ax = plt.subplots(figsize=(22, 6))
im = ax.imshow(frac, cmap="viridis", vmin=0, vmax=1, aspect="auto")
for i in range(frac.shape[0]):
    for j in range(frac.shape[1]):
        if inter[i, j] > 0:
            ax.text(j, i, f"{100 * frac[i, j]:.0f}%\n{fmt_weight(inter[i, j])}", ha="center", va="center",
                    fontsize=7, color="white" if frac[i, j] < 0.6 else "black")
ax.set_xticks(range(len(bdt_names)))
ax.set_xticklabels([n.removeprefix("bdt_") for n in bdt_names], rotation=60, ha="right")
ax.set_yticks(range(len(wc_names)))
ax.set_yticklabels([f"{labels[n]} ({fmt_weight(weights[membership[n]].sum())})" for n in wc_names])
ax.set_xlabel("BAGELS selection")
ax.set_title("Where the WC selections' events go in the BAGELS categorization (weighted prediction, full data POT)")
fig.colorbar(im, ax=ax, fraction=0.02, label="fraction of WC selection")
save(fig, "overlap_matrix_wc_to_bagels_categories")
plt.show()